(user-foundations-performance-benchmarks)=
# Benchmarks

Empirical performance metrics and throughput comparisons show MolSysMT's scaling behavior and comparative execution speed against **MDTraj**, **MDAnalysis**, and **SciPy** where the measured operation uses it.

## Competitive Performance Timing Matrix

The table below presents median execution durations from the committed competitive baseline. It is rendered from the same published JSON consumed by the interactive developer dashboard, using the Chicken Villin HP35 dataset:

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML

baseline_path = Path("../../../../_static/benchmarks_data/competitor_matrix_session.json")
results = json.loads(baseline_path.read_text())["results"]

def milliseconds(key):
    return f"{results[key]['median_seconds'] * 1000:.2f} ms"

data = [
    {"Operation Area": "Trajectory Load (DCD)", "MolSysMT Public": milliseconds("competitor_loading_molsysmt"), "MolSysMT Native Kernel": "N/A", "MDTraj / SciPy": milliseconds("competitor_loading_mdtraj"), "MDAnalysis": milliseconds("competitor_loading_mdanalysis")},
    {"Operation Area": "Selection Simple (CA)", "MolSysMT Public": milliseconds("competitor_selection_molsysmt_simple"), "MolSysMT Native Kernel": "N/A", "MDTraj / SciPy": milliseconds("competitor_selection_mdtraj_simple"), "MDAnalysis": milliseconds("competitor_selection_mdanalysis_simple")},
    {"Operation Area": "Selection Complex", "MolSysMT Public": milliseconds("competitor_selection_molsysmt_complex"), "MolSysMT Native Kernel": "N/A", "MDTraj / SciPy": milliseconds("competitor_selection_mdtraj_complex"), "MDAnalysis": milliseconds("competitor_selection_mdanalysis_complex")},
    {"Operation Area": "Center of Geometry", "MolSysMT Public": milliseconds("competitor_center_molsysmt_public"), "MolSysMT Native Kernel": milliseconds("competitor_center_molsysmt_jit"), "MDTraj / SciPy": milliseconds("competitor_center_mdtraj"), "MDAnalysis": milliseconds("competitor_center_mdanalysis")},
    {"Operation Area": "RMSD Calculation", "MolSysMT Public": milliseconds("competitor_rmsd_molsysmt_public"), "MolSysMT Native Kernel": milliseconds("competitor_rmsd_molsysmt_jit"), "MDTraj / SciPy": milliseconds("competitor_rmsd_mdtraj"), "MDAnalysis": milliseconds("competitor_rmsd_mdanalysis")},
    {"Operation Area": "Pairwise Distances", "MolSysMT Public": milliseconds("competitor_distances_molsysmt_public"), "MolSysMT Native Kernel": milliseconds("competitor_distances_molsysmt_jit"), "MDTraj / SciPy": milliseconds("competitor_distances_mdtraj"), "MDAnalysis": milliseconds("competitor_distances_mdanalysis")}
]

df = pd.DataFrame(data)
html_table = df.to_html(classes="table", index=False)
html_table = html_table.replace('<th>', '<th style="text-align: left;">').replace('<td>', '<td style="text-align: left;">')
HTML(html_table)

Operation Area,MolSysMT Public,MolSysMT Native Kernel,MDTraj / SciPy,MDAnalysis
Trajectory Load (DCD),19.28 ms,N/A,20.88 ms,54.96 ms
Selection Simple (CA),1.95 ms,N/A,2.73 ms,0.17 ms
Selection Complex,4.35 ms,N/A,16.94 ms,0.64 ms
Center of Geometry,49.65 ms,35.23 ms,1.43 ms,144.46 ms
RMSD Calculation,35.69 ms,29.16 ms,0.21 ms,143.62 ms
Pairwise Distances,6.73 ms,2.35 ms,0.30 ms,141.51 ms


## Key Architectural Observations

- **One comparable snapshot**: Every value in the table comes from one committed benchmark session, so its environment and repetition metadata remain inspectable.
- **Hardware-dependent evidence**: Use these measurements to compare operations within this recorded environment. Re-run the suite before making decisions for different hardware or dependency versions.
- **Validated public boundaries**: Public MolSysMT calls include argument validation and physical-unit handling. Native-kernel rows isolate lower-level numerical work where the operation has a directly comparable kernel.